# Exercise 2. LoRa for Low-Resource Languages
NLP for social good involves not only reducing harmful outputs but also making AI accessible across languages, including low- and medium-resource ones like Nigerian Pidgin and Danish.


```{figure} ../figures/class8/neural-space-low-resource.png
---
name: neural-space-low-resource
width: 100%
---
borrowed from [NeuralSpace blogpost](https://medium.com/neuralspace/challenges-in-using-nlp-for-low-resource-languages-and-how-neuralspace-solves-them-54a01356a71b) by Felix Laumann
```

Fine-tuning LLMs can help, but it is costly. LoRA (Low-Rank Adaptation) offers a parameter-efficient alternative, reducing trainable parameters by up to 10,000 times. In other words, rather than training all 8 billion parameters of a model like [Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B-Base), LoRA updates only a small fraction. This also reduces environmental costs!

## 2.1 Intro to LoRa?
If you're interested in the math behind LoRa (but in an intuitive way), I encourage you to read Sebastian Raschka's [blogpost](https://magazine.sebastianraschka.com/i/138081202/a-brief-introduction-to-lora). You can also read the original paper by {cite:t}`hu_lora_2021`. 

```{figure} ../figures/class8/lora_adapter.png
---
name: lora_adapter
width: 100%
---
From HF's [smol course](https://huggingface.co/learn/smol-course/en/unit1/3a)
```

In the code implementation ([PEFT](https://huggingface.co/docs/peft/index) library), LoRa is treated as a sort of "adapter" that you can train and keep seperately, essentially allowing you to place it on other models (if the base architecture matches):

:::{admonition} What are Adapters? Is LoRa Really an Adapter?
:class: dropdown, tip
Adapters are extra trainable parameters that you add to a model, keeping its own model weights frozen. While the original paper does not call LoRa an adapter {cite:p}`hu_lora_2021`, the term is used everywhere. 

Read Jason Phang's take on this: [Should we consider LoRa an Adapter?](https://jasonphang.com/posts/2023/07/post1/).   
-> TL:DR; Historically, LoRa is not an adapter, but it probably can be considered one.
:::

## 2.2 Setup
For the code implementation, we'll use the [PEFT](https://huggingface.co/docs/peft/en/index) and [TRL](https://huggingface.co/docs/trl/en/index) library by Hugging Face
```bash
source .venv/bin/activate
pip install peft trl
```

If you don't already have this in your `venv`, we also need:
```bash
pip install transformers datasets torch
```

Let's import:

In [112]:
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from datasets import load_dataset
from peft import LoraConfig, AutoPeftModelForCausalLM
import torch

from trl import SFTTrainer, SFTConfig

## 2.3 Load Model & Data
In today's exercise, we'll try to improve `SmolLM2-135M-Instruct`'s ability to translate from English to Danish.

:::{admonition}  LoRa can do much more than Translation :)
:class: dropdown, tip
As a simple introduction to LoRA, we're doing machine translation, but you can use this approach for anything you'd like really - feel free to switch out the dataset for something you'd like. Or use this notebook as a inspiration for the exam :).

See also this tutorial for instruction-tuning a danish language model using QLoRA -> [Tutorial: Finetuning Language Models](https://www.foundationmodels.dk/blog/2024/02/02/tutorial-finetuning-language-models.html)
:::

We'll start by loading `SmolLM2` (NB. it is smaller than in [Exercise 1](/book/class8/001_steering_vectors.ipynb)!)

In [113]:
model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_id)

We'll load a Danish-English translation dataset, but only a subset with `[:n]` for `n` rows:

In [114]:
n_rows = 2000
train_ds = load_dataset("Helsinki-NLP/opus-100", "da-en", split=f"train[:{n_rows}]")

Let's look at the `translation` column:

In [115]:
train_ds["translation"]

Column([{'da': 'På Det Blandede EØS-Udvalgs vegne', 'en': 'For the EEA Joint Committee'}, {'da': 'metal, der indeholder mindst 99,9 vægtprocent bly, forudsat ingen anden bestanddel indgår i mængder, der overstiger de i nedenstående skema anførte grænseværdier:', 'en': 'Metal containing by weight at least 99,9 % of lead, provided that the content by weight of any other element does not exceed the limit specified in the following table:'}, {'da': 'Tænk.', 'en': 'Think.'}, {'da': 'Beth...', 'en': 'Beth...'}, {'da': 'Vort projekt "Menneskelig dvale" gør det muligt at holde vores bedste mænd nedfrosset i deres bedste tilstand, for at bruge dem efter behov.', 'en': 'With the Human Hibernation Project, we will be able to save our best men... frozen in their prime, for use when they are needed most.'}])

Let's print a few:

In [116]:
for translation in train_ds["translation"][:3]:
    print(f"EN: {translation['en']}")
    print(f"DA: {translation['da']}")
    print()

EN: For the EEA Joint Committee
DA: På Det Blandede EØS-Udvalgs vegne

EN: Metal containing by weight at least 99,9 % of lead, provided that the content by weight of any other element does not exceed the limit specified in the following table:
DA: metal, der indeholder mindst 99,9 vægtprocent bly, forudsat ingen anden bestanddel indgår i mængder, der overstiger de i nedenstående skema anførte grænseværdier:

EN: Think.
DA: Tænk.



## 2.4 Chat Templating
Last week, we played with chat template formatting like this:
```{figure} ../figures/class8/messages_SFT_lora.png
---
name: messages_sft_lora
width: 80%
---
`Messages` dictionary with chat template
```

We want something similar for the task of translating English to Danish. That is, we want a `prompt` containing instructions + an English example & a `desired_response` being the `Danish` example.

### Your Turn: Define a Prompt + Create Formatting Function
:::{admonition} HANDS-ON
:class: red
1. Create a function called `def format_prompt(example)`
    - It should process a single row `example` in our dataset
    - Define a prompt that contains instructions and an English example. 
    - Format it as a `messages` dictionary, where the `desired_response` is the `Danish` translation.
    - Return the dictionary 

2. Test the function on a single example in `train_ds` & print the result.

3. Use the `map` function (like we did in [class 5](/book/class5/001_finetune.ipynb) when tokenizing) on your `train_ds`, returning `formatted_train_ds`.
:::

#### Solution
Preprocess function (1) + printing one example (2):

In [117]:
def format_prompt(example):
    translation = example["translation"]
    return {"messages": [{"role": "user", "content": f"Translate to Danish: {translation['en']}"}, {"role": "assistant", "content": f"{translation['da']}"}]}

# print one example with train_ds
example = train_ds[0]
formatted_example = format_prompt(example)
print(formatted_example)


{'messages': [{'role': 'user', 'content': 'Translate to Danish: For the EEA Joint Committee'}, {'role': 'assistant', 'content': 'På Det Blandede EØS-Udvalgs vegne'}]}


Use the `map` function (3):

In [118]:
formatted_train_ds = train_ds.map(format_prompt, batched=False)

### Final Formatting: Removing the `Translation` Col
We should now have this:

In [119]:
formatted_train_ds

Dataset({
    features: ['translation', 'messages'],
    num_rows: 2000
})

We remove the `translation` col:

In [120]:
formatted_train_ds = formatted_train_ds.remove_columns(["translation"])

## 2.5 LoRa Config & Training
We'll start by configuring LoRA: 

In [121]:
rank = 16
peft_config = LoraConfig(
    r=rank,
    lora_alpha=rank * 2,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

:::{admonition} HANDS-ON
:class: red
Look at the LoRa hyperparameters [here](https://docs.unsloth.ai/get-started/fine-tuning-llms-guide/lora-hyperparameters-guide#hyperparameters-and-recommendations), and see if they make sense to you. Google around otherwise!
:::

Let's define a output_dir:

In [122]:
path = Path.cwd()
output_dir = path.parents[0] / "training" / f".smollm2_da_en_{n_rows}" #n-rows for different train sizes

### Train
We are ready to train:

In [123]:
trainer = SFTTrainer(
    model=model,
    args=SFTConfig(
        output_dir=output_dir,
        overwrite_output_dir=True,
        num_train_epochs=1,
        per_device_train_batch_size=2,
        packing=True, # can speed up training
        chat_template_path=model_id, # use the model's built-in chat template (since we're not tokenizing ourselves)
    ),
    train_dataset=formatted_train_ds,
    peft_config=peft_config
)
trainer.train()

Padding-free training is enabled, but the attention implementation is not set to a supported flash attention variant. Padding-free training flattens batches into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn, kernels-community/flash-attn3, kernels-community/vllm-flash-attn3. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation` in the model configuration to one of these supported options or verify that your attention mechanism can handle flattened sequences.
You are using packing, but the attention implementation is not set to a supported flash attention variant. Packing gathers multiple samples into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn, kernels-community/flash-attn3, kernels-community/vllm-fla

Step,Training Loss
10,4.240900
20,4.258900
30,4.190400
40,4.105400
50,4.006600
60,4.079700


TrainOutput(global_step=60, training_loss=4.146974690755209, metrics={'train_runtime': 78.8303, 'train_samples_per_second': 1.522, 'train_steps_per_second': 0.761, 'total_flos': 77257304886528.0, 'train_loss': 4.146974690755209, 'epoch': 1.0})

## 2.6 Inference! Test our LoRa :)
Since we specified an `output_dir`, the lora adapter has been saved in various checkpoints. We pick the latest:

In [124]:
path = Path.cwd()
output_dir = path.parents[0] / "training" / f".smollm2_da_en_{n_rows}"
adapter_path = str(output_dir) + "/checkpoint-60"

We load the model with the adapter:

In [131]:
tokenizer = AutoTokenizer.from_pretrained(adapter_path, local_files_only=True)
model = AutoPeftModelForCausalLM.from_pretrained(adapter_path, device_map="auto", torch_dtype=torch.float16, local_files_only=True)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

Device set to use mps


We can check the model name and PEFT config:

In [132]:
print(f"Model: {model.config._name_or_path}")
print(f"PEFT config: {model.peft_config}")

Model: HuggingFaceTB/SmolLM2-135M-Instruct
PEFT config: {'default': LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path='HuggingFaceTB/SmolLM2-135M-Instruct', revision=None, inference_mode=True, r=16, target_modules={'q_proj', 'v_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None)}


### Translate

In [127]:
prompt = "Translate to Danish: I love to drive my car."
format_prompt = pipe.tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
outputs = pipe(format_prompt, max_new_tokens=50, return_full_text=False)
print(outputs[0]["generated_text"])

Du vite dette kommerliga kan.


:::{admonition} QUESTION
:class: red
If you don't speak Danish, try to put the answer below into google translate (or ask a Danish speaking friend). Consider if this answer is good.
:::

## 2.7 TV Kitchen: Inference with More Training Examples
Perhaps 2000 examples was not enough to train a proper translation machine. For the sake of this class, I didn't want to make you wait 10-30 minutes for training with more examples. 

However, like in a TV-kitchen, I have pre-made LoRa's for you to try in the `resources` folder:

<div style="display:flex; justify-content:center;">
  <div style="background:white; border:1px solid black; padding:6px; display:inline-block; font-family:monospace; white-space:pre;">&gt; .smollm2_da_en_2000 / checkpoint-60
&gt; .smollm2_da_en_10000 / checkpoint-357
&gt; .smollm2_da_en_20000 / checkpoint-730
&gt; .smollm2_da_en_50000 / checkpoint-1849</div>
</div>

:::{admonition} Adhere to my prompt!
:class: important, dropdown
While you might have defined your own instructions, please note that my prompt is:
```python
prompt = "Translate to Danish: English sentence"
```
In this TV kitchen, it might affect performance if you deviate from this way of prompting. However, if you are curious, feel free to test if another prompt still works :). You can try whatever English sentence you'd like.
:::

### Your Turn: Test Inference on Various LoRas

:::{admonition} HANDS-ON
:class: red
1. **Make a function `def test_inference(adapter_path, prompt)`:**
    - Takes an `adapter_path` pointing to one of the pre-made LoRAs 
    - Takes a prompt with an instuction and an English sentence (of your choice) to translate 
    - Does all of the inference steps above (loading tokenizer, peft model, taking the prompt and applying chat template to it, returns output ...)
<br><br>
2. Test your function on the various LoRa adapters :)
:::

To get you started, you should fix these paths:

In [128]:
# fix this with n_rows and correct checkpoint path :)
resources_dir = path.parents[1] / "resources" / "models" / "lora" / f".smollm2_da_en_{n_rows}" #n-rows for different train sizes
adapter_path = str(resources_dir) + "/"

(optionally, find a smart way to get the last `checkpoint` ..)

### Solution

In [129]:
def test_inference(adapter_path, prompt):
    # load model with adapter
    print(f"[INFO:] Loading adapter")
    tokenizer = AutoTokenizer.from_pretrained(adapter_path, local_files_only=True)
    model = AutoPeftModelForCausalLM.from_pretrained(adapter_path, device_map="auto", torch_dtype=torch.float16, local_files_only=True)

    print(f"[INFO:] Loading pipeline")
    pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

    format_prompt = pipe.tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)

    print(f"[INFO:] Generating translation")
    outputs = pipe(format_prompt, max_new_tokens=50, return_full_text=False)
    return outputs[0]["generated_text"]

# try it 
n_rows = 20000
resources_dir = path.parents[1] / "resources" / "models" / "lora" / f".smollm2_da_en_{n_rows}" #n-rows for different train sizes
adapter_path = str(resources_dir) + "/checkpoint-730"

# test inference
prompt = "Translate to Danish: The weather is nice today."
translation = test_inference(adapter_path, prompt)
print( f"Translation: {translation}" )

[INFO:] Loading adapter


Device set to use mps


[INFO:] Loading pipeline
[INFO:] Generating translation
Translation: Danske værendige



## 2.8 Future work
To build on this, we could have:
1. Experimented with the prompt format. I ran a simple `Translate to Danish`. Maybe it wasn't ideal. 
2. Done benchmarking such as evaluating on a `test set` rather than `vibe-checking` if the LoRa became better as we increased training examples
    - This is more relevant when we have a better working version :)
3. Tried different parameters for both LoRa and the actual training, especially epochs (see also green tip box below).
4. Tried a different LLM.
5. Tried running inference with temperature = 0 (for no stochasticity)

:::{admonition} How do I select parameters for tuning LoRa?
:class: tip, dropdown
It often requires some experimenting to find the right parameters. We can also draw on other people's experiences with the caveat that their context might not translate to ours.   

You might like Sebastian Raschka's [blogpost](https://magazine.sebastianraschka.com/p/practical-tips-for-finetuning-llms) (also the one linked in the beginning of this exercise).
:::